In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.1 Generate text

> 🎯 **Goal:** train a tiny language model on Shakespeare, ask it to continue a line, and watch real generated text appear.

**Why this matters.** This is the whole course in miniature. By the end of PocketLM you will have built every piece of a GPT-style model by hand. Today you skip to the payoff: you run a working model so you can see, with your own eyes, what we are building toward. Everything later (tokens, probabilities, attention, training) is just opening up the box you are about to shake.

**The intuition.** A language model is an extremely well-read autocomplete. You give it a few characters, and it guesses what comes next, then guesses again using its own guess, over and over. Picture handing someone the start of a sentence and asking them to keep going one letter at a time. If they have read a lot of Shakespeare, the letters they add will *look* like Shakespeare even before they form real words. Our model has read a little Shakespeare and trained for only a moment, so it produces the look of English before the meaning of English.

**The mechanics.** A few terms you will meet in the code. A *corpus* is the training text (here, Shakespeare). *Training* is the process of adjusting the model's internal numbers so its guesses match the corpus better. A *token* is the unit the model reads and writes (for us, a single character). *Generation* means: feed the model a starting string (the *prompt*), let it produce one token, append it, and repeat until we have enough. The model below has just 2 layers and trains for only a handful of steps, so do not expect sense. Expect *shape*: line breaks, capitalized speaker names like ROMEO, sensible spacing. That shape is the model genuinely learning the statistics of the text.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
out = generate(model, tok, "ROMEO:", max_new_tokens=200, seed=0)
print(out)

**What just happened.** You trained a neural network and it wrote you 200 characters of pseudo-Shakespeare, starting from the prompt `ROMEO:`. It is mostly nonsense, but look closer: the line breaks fall in plausible places, words have plausible lengths, capital letters cluster where speaker names go. Even a 2-layer model trained for a handful of steps reproduces the *texture* of the corpus. Nobody told it any of these rules. It absorbed them from the raw text.

⚠️ **Common pitfalls**
- Expecting it to make sense. It will not yet. Coherent meaning needs a bigger model and far more training; right now you are watching the *style* emerge before the substance.
- Thinking the output is random. It is not. We fixed `seed=0`, so re-running gives the exact same text. That reproducibility is what lets the later lessons compare results fairly.

🏋️ **Try it yourself.** Change the prompt `"ROMEO:"` to `"JULIET:"` or `"The "` and re-run. Notice the model still produces the same overall texture no matter how you start it. Then bump `max_new_tokens=200` up to `500` and watch it keep going.

The CI guard below simply confirms the model behaved: it produced exactly the number of characters we asked for, and every character it emitted is one it actually knows. The next lessons open up *why* all this works: tokens, probabilities, and how decoding turns probabilities into text.

In [ ]:
# CI guard: generation never changes length or emits non-vocab characters.
assert len(out) == len("ROMEO:") + 200
assert set(out) <= set(tok.itos)